In [16]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import warnings
warnings.filterwarnings("ignore")

BASE     = '/content/drive/MyDrive/DS340 Final Project'
REVIEWS  = f'{BASE}/Yelp JSON/yelp_dataset/yelp_academic_dataset_review.json'
BIZ      = f'{BASE}/Yelp JSON/yelp_dataset/yelp_academic_dataset_business.json'

df_train = pd.read_parquet(f'{BASE}/split_train.parquet')
df_val   = pd.read_parquet(f'{BASE}/split_val.parquet')
df_test  = pd.read_parquet(f'{BASE}/split_test.parquet')

KEYWORDS = df_train["keyword"].unique()
CITIES   = df_train["city"].unique()

print(f"Train: {len(df_train)} rows")
print(f"Val:   {len(df_val)} rows")
print(f"Test:  {len(df_test)} rows")
print(f"Keywords: {list(KEYWORDS)}")
print(f"Cities:   {list(CITIES)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train: 5112 rows
Val:   216 rows
Test:  1116 rows
Keywords: ['birria', 'birria tacos', 'boba', 'charcuterie', 'elote', 'korean fried chicken', 'matcha', 'shakshuka', 'smash burger', 'truffle fries', 'tteokbokki', 'wagyu']
Cities:   ['Nashville', 'Philadelphia', 'Tampa']


In [17]:
!pip install transformers torch --quiet

In [18]:
import json

BUSINESSES = "/content/drive/MyDrive/DS340 Final Project/Yelp JSON/yelp_dataset/yelp_academic_dataset_business.json"
CITIES_SET = {"Philadelphia", "Tampa", "TAMPA", "Nashville"}

businesses = []
with open(BUSINESSES) as f:
    for line in f:
        b = json.loads(line)
        if b.get("city") in CITIES_SET and "Restaurants" in (b.get("categories") or ""):
            businesses.append({"business_id": b["business_id"], "city": b["city"]})

biz_df = pd.DataFrame(businesses)
biz_df["city"] = biz_df["city"].replace("TAMPA", "Tampa")

valid_ids  = set(biz_df["business_id"])
id_to_city = dict(zip(biz_df["business_id"], biz_df["city"]))

print(f"Found {len(biz_df)} restaurants")
print(biz_df["city"].value_counts())

Found 11321 restaurants
city
Philadelphia    5852
Tampa           2967
Nashville       2502
Name: count, dtype: int64


In [19]:
FOOD_LEXICON = [
    "birria", "smash burger", "boba", "matcha", "birria tacos",
    "korean fried chicken", "charcuterie", "wagyu", "truffle fries",
    "elote", "tteokbokki", "shakshuka", "cloud eggs", "butter board"
]

REVIEWS = "/content/drive/MyDrive/DS340 Final Project/Yelp JSON/yelp_dataset/yelp_academic_dataset_review.json"

# This collects the actual review texts grouped by city+month+keyword
# We cap at 50 reviews per group to keep embedding time manageable
MAX_REVIEWS_PER_GROUP = 50

from collections import defaultdict

review_groups = defaultdict(list)  # key: (city, month, keyword)

with open(REVIEWS) as f:
    for line in f:
        r = json.loads(line)
        if r["business_id"] not in valid_ids:
            continue
        text = r["text"].lower()
        date = r["date"][:7]
        city = id_to_city[r["business_id"]]

        for kw in FOOD_LEXICON:
            if kw in text:
                key = (city, date, kw)
                if len(review_groups[key]) < MAX_REVIEWS_PER_GROUP:
                    review_groups[key].append(text)

print(f"Total city+month+keyword groups with reviews: {len(review_groups)}")
print("Sample key:", list(review_groups.keys())[0])

Total city+month+keyword groups with reviews: 2818
Sample key: ('Philadelphia', '2014-09', 'truffle fries')


In [20]:
import torch
import numpy as np
from transformers import DistilBertTokenizer, DistilBertModel

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
model     = DistilBertModel.from_pretrained("distilbert-base-uncased")
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Using device: {device}")

def embed_texts(texts, batch_size=16):
    """Embed a list of texts and return the mean pooled vector."""
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)
        with torch.no_grad():
            output = model(**encoded)
        # Mean pool over token dimension
        embeddings = output.last_hidden_state.mean(dim=1).cpu().numpy()
        all_embeddings.append(embeddings)
    return np.vstack(all_embeddings).mean(axis=0)  # mean over all reviews in group

# Embed all groups
embedding_records = []

total = len(review_groups)
for idx, ((city, month, kw), texts) in enumerate(review_groups.items()):
    if idx % 50 == 0:
        print(f"Embedding group {idx}/{total}...")
    vec = embed_texts(texts)
    embedding_records.append({
        "city":    city,
        "month":   month,
        "keyword": kw,
        "embedding": vec
    })

print(f"Done. Total embeddings: {len(embedding_records)}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Using device: cuda
Embedding group 0/2818...
Embedding group 50/2818...
Embedding group 100/2818...
Embedding group 150/2818...
Embedding group 200/2818...
Embedding group 250/2818...
Embedding group 300/2818...
Embedding group 350/2818...
Embedding group 400/2818...
Embedding group 450/2818...
Embedding group 500/2818...
Embedding group 550/2818...
Embedding group 600/2818...
Embedding group 650/2818...
Embedding group 700/2818...
Embedding group 750/2818...
Embedding group 800/2818...
Embedding group 850/2818...
Embedding group 900/2818...
Embedding group 950/2818...
Embedding group 1000/2818...
Embedding group 1050/2818...
Embedding group 1100/2818...
Embedding group 1150/2818...
Embedding group 1200/2818...
Embedding group 1250/2818...
Embedding group 1300/2818...
Embedding group 1350/2818...
Embedding group 1400/2818...
Embedding group 1450/2818...
Embedding group 1500/2818...
Embedding group 1550/2818...
Embedding group 1600/2818...
Embedding group 1650/2818...
Embedding group 17

In [21]:
embed_df = pd.DataFrame(embedding_records)

# Expand embedding vector into separate columns
embed_cols = pd.DataFrame(
    np.vstack(embed_df["embedding"].values),
    columns=[f"emb_{i}" for i in range(768)]
)
embed_df = pd.concat([embed_df.drop(columns="embedding"), embed_cols], axis=1)

# Merge with trend scores from splits
TRAIN_END = "2018-12"
VAL_END   = "2019-06"

embed_df["split"] = "test"
embed_df.loc[embed_df["month"] <= TRAIN_END, "split"] = "train"
embed_df.loc[(embed_df["month"] > TRAIN_END) & (embed_df["month"] <= VAL_END), "split"] = "val"

# Merge trend score targets
target_df = pd.concat([df_train, df_val, df_test])[["city", "month", "keyword", "trend_score_smooth"]]
embed_df  = embed_df.merge(target_df, on=["city", "month", "keyword"], how="inner")

print(f"Feature matrix shape: {embed_df.shape}")
print(f"Split counts:\n{embed_df['split'].value_counts()}")

Feature matrix shape: (2818, 773)
Split counts:
split
train    1837
test      841
val       140
Name: count, dtype: int64


In [22]:
#save embed_df to drive
BASE = '/content/drive/MyDrive/DS340 Final Project'
embed_df.to_parquet(f'{BASE}/embed_df.parquet', index=False)
print('Saved embed_df.parquet to Drive')

Saved embed_df.parquet to Drive


In [23]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

emb_cols = [c for c in embed_df.columns if c.startswith("emb_")]

X_train = embed_df[embed_df["split"] == "train"][emb_cols].values
y_train = embed_df[embed_df["split"] == "train"]["trend_score_smooth"].values

X_val   = embed_df[embed_df["split"] == "val"][emb_cols].values
y_val   = embed_df[embed_df["split"] == "val"]["trend_score_smooth"].values

X_test  = embed_df[embed_df["split"] == "test"][emb_cols].values
y_test  = embed_df[embed_df["split"] == "test"]["trend_score_smooth"].values

# Ridge regression to handle high-dimensional embeddings
reg = Ridge(alpha=1.0)
reg.fit(X_train, y_train)

val_mae  = mean_absolute_error(y_val,  reg.predict(X_val))
test_mae = mean_absolute_error(y_test, reg.predict(X_test))

print(f"DistilBERT + Linear Regression")
print(f"  Val  MAE: {val_mae:.6f}")
print(f"  Test MAE: {test_mae:.6f}")

DistilBERT + Linear Regression
  Val  MAE: 0.000840
  Test MAE: 0.001170


In [24]:
test_df = embed_df[embed_df["split"] == "test"].copy()
test_df["predicted"] = reg.predict(X_test)
test_df["abs_error"] = (test_df["predicted"] - test_df["trend_score_smooth"]).abs()

# Drop embedding columns before saving
save_cols = ["city", "month", "keyword", "trend_score_smooth", "predicted", "abs_error"]
test_df[save_cols].to_parquet("distilbert_predictions.parquet", index=False)

files.download("distilbert_predictions.parquet")
print("Downloaded distilbert_predictions.parquet")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded distilbert_predictions.parquet
